# INbreast: jedna dojka, CC/MLO par

Leva i desna dojka su odvojeni primeri. Pacijent grupiše sve datume/strane radi sprečavanja leakage-a; nema patient-level predikcije. Notebook koristi aktuelni source paket, metadata schema 2 i preprocessing 2. Smoke nije finalni rezultat.


## Aktuelni kod

Učitajte `INbreast_source.zip` iz ove verzije projekta ili postavite `INBREAST_PROJECT_DIR` na već prenet aktuelni checkout. Dataset nije uključen u source paket.


In [ ]:
import os, sys, subprocess, zipfile, json
from pathlib import Path
PROJECT_DIR = Path(os.environ.get('INBREAST_PROJECT_DIR', '/content/INBreast'))
if not (PROJECT_DIR / 'train.py').is_file():
    from google.colab import files
    uploaded = files.upload()
    archives = [Path(name) for name in uploaded if name.endswith('.zip')]
    if len(archives) != 1:
        raise ValueError('Učitajte jedan aktuelni INbreast_source.zip.')
    PROJECT_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archives[0]) as archive:
        for member in archive.infolist():
            target = (PROJECT_DIR / member.filename).resolve()
            if not target.is_relative_to(PROJECT_DIR.resolve()):
                raise ValueError('Nevalidna putanja u ZIP-u.')
        archive.extractall(PROJECT_DIR)
os.chdir(PROJECT_DIR)
os.environ['HF_HOME'] = str(PROJECT_DIR / 'artifacts' / 'model_cache')
os.environ['MPLCONFIGDIR'] = '/tmp/inbreast-matplotlib'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-dev.txt'], check=True)


## GPU, okruženje i kompletni testovi

CUDA kombinaciju dokumentuje stvarni runtime; lokalna CUDA kombinacija nije verifikovana. Ako Colab instalira drugi PyTorch wheel, zabeležite verzije pre eksperimenta.


In [ ]:
import torch
subprocess.run([sys.executable, 'environment.py'], check=True)
subprocess.run(['nvidia-smi'], check=False)
if not torch.cuda.is_available():
    raise RuntimeError('Za finalni Colab trening uključite GPU runtime.')
subprocess.run([sys.executable, '-m', 'pytest'], check=True)
from data import METADATA_SCHEMA_VERSION
from preprocessing import PREPROCESSING_VERSION
assert METADATA_SCHEMA_VERSION == 2 and PREPROCESSING_VERSION == 2


## Putanje i konfiguracija

Promenljive su izmenjive; prenesite dataset na Drive ili postavite drugu putanju. Klinička labela ostaje radiološki proxy: BI-RADS 1–3 = 0 i 4–6 = 1.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATA_ROOT = Path(os.environ.get('INBREAST_DATA_ROOT', '/content/drive/MyDrive/INbreast Release 1.0'))
OUTPUT = Path(os.environ.get('INBREAST_OUTPUT', '/content/drive/MyDrive/INbreast_artifacts'))
if not DATA_ROOT.is_dir():
    raise FileNotFoundError(f'Podesite DATA_ROOT: {DATA_ROOT}')
OUTPUT.mkdir(parents=True, exist_ok=True)
import yaml
config = yaml.safe_load(Path('config.yaml').read_text(encoding='utf-8'))
config.update(data_root=str(DATA_ROOT), output_dir=str(OUTPUT), metadata_pairs=str(OUTPUT / 'metadata_pairs.csv'),
              cache_dir='/content/inbreast_preprocessed', workers=2, device='cuda')
CONFIG = OUTPUT / 'colab_config.yaml'
CONFIG.write_text(yaml.safe_dump(config, sort_keys=False), encoding='utf-8')
def run(mode, output=OUTPUT, *extra):
    command = [sys.executable, 'train.py', '--config', str(CONFIG), '--mode', mode, '--output-dir', str(output), *extra]
    print(' '.join(command))
    subprocess.run(command, check=True)


## Prepare, audit i sanity

Metadata se automatski generiše samo u prepare režimu. Svi ostali režimi učitavaju istu tabelu. Nepotpuni parovi i isključene slike imaju zasebne audit CSV-ove.


In [ ]:
run('prepare')
import pandas as pd
from IPython.display import Image, display
pairs = pd.read_csv(OUTPUT / 'metadata_pairs.csv', dtype={'patient_id': str, 'acquisition_date': str})
assert pairs.metadata_schema_version.eq(2).all() and pairs.preprocessing_version.eq(2).all()
assert pairs.groupby('patient_id').split.nunique().eq(1).all()
display(pd.read_csv(OUTPUT / 'dataset_summary.csv'))
display(pd.read_csv(OUTPUT / 'incomplete_pairs.csv'))
display(pd.read_csv(OUTPUT / 'excluded_images.csv'))
print((OUTPUT / 'split_audit.json').read_text())
run('sanity')
display(Image(filename=str(OUTPUT / 'sanity' / 'paired_overlays.png')))


## Kratak smoke i resume

Dve epohe i nastavak na treću proveravaju checkpoint/resume/evaluate/predict. To nije konačna procena. Smoke koristi odvojen direktorijum i ne daje težine finalnom eksperimentu.


In [ ]:
SMOKE = OUTPUT / 'smoke'
run('train', SMOKE, '--size', '224', '--epochs', '2', '--no-pretrained')
run('train', SMOKE, '--size', '224', '--epochs', '3', '--no-pretrained', '--resume', str(SMOKE / 'last.pt'))
run('evaluate', SMOKE, '--eval-split', 'val', '--bootstrap-iterations', '100')
run('predict', SMOKE, '--pair-id', str(pairs.iloc[0].pair_id))
run('predict', SMOKE, '--cc-path', str(pairs.iloc[0].cc_dicom_path), '--mlo-path', str(pairs.iloc[0].mlo_dicom_path))


## Finalni trening od početka i grouped evaluacija

Postavite RUN_FINAL=True tek kada testovi, metadata, leakage, sanity i smoke/resume prođu. Outer test fold ne učestvuje u izboru checkpointa, praga ni kalibratora. CV koristi nezavisne unutrašnje holdout skupove, bez pretrage hiperparametara; nije puna nested hyperparameter CV.


In [ ]:
RUN_FINAL = False
FINAL_HOLDOUT = OUTPUT / 'final_holdout'
FINAL_CV = OUTPUT / 'final_baseline'
if RUN_FINAL:
    run('train', FINAL_HOLDOUT, '--epochs', '25')
    run('evaluate', FINAL_HOLDOUT)
    run('crossval', FINAL_CV, '--folds', '5', '--epochs', '25')
    display(pd.read_csv(FINAL_CV / 'crossval' / 'fold_metrics.csv'))
    print((FINAL_CV / 'crossval' / 'metrics.json').read_text())
else:
    print('Finalni trening nije pokrenut. Smoke metrike nisu finalni rezultat.')


## Resume finalnog holdout treninga

Koristite istu konfiguraciju i metadata. Ne nastavljajte stari/nekompatibilni checkpoint. Ne povećavajte epohe ili birajte druge parametre prema već viđenim test rezultatima.


In [ ]:
RESUME_HOLDOUT = False
if RESUME_HOLDOUT:
    run('train', FINAL_HOLDOUT, '--epochs', '25', '--resume', str(FINAL_HOLDOUT / 'last.pt'))


## Predikcija, ograničen Grad-CAM i analiza grešaka

Prijavljena verovatnoća ima raw/calibrated oznaku. Grad-CAM heatmap i thresholded Grad-CAM region su post-hoc prikazi, nisu segmentacija. Lokalizacija je u originalnoj rezoluciji, sa paddingom isključenim i samo validnim ROI-jem. Preklapanje sa ROI-jem ne potvrđuje klinički ispravne osobine. Vizuelno proverite tekst, ivice, pozadinu, artefakte, pektoralni mišić i crop.


In [ ]:
RUN_EXPLANATIONS = False
if RUN_EXPLANATIONS:
    run('evaluate', FINAL_HOLDOUT, '--gradcam-enabled', '--gradcam-examples', '8', '--gradcam-categories', 'FN', 'FP')
    run('predict', FINAL_HOLDOUT, '--pair-id', str(pairs.iloc[0].pair_id), '--gradcam-enabled', '--gradcam-examples', '1')
    print((FINAL_HOLDOUT / 'error_analysis.json').read_text())
    display(Image(filename=str(FINAL_HOLDOUT / 'reliability_diagram.png')))
    for path in sorted((FINAL_HOLDOUT / 'comparisons').glob('*.png'))[:4]:
        display(Image(filename=str(path), width=1000))
